# Notebook 2: Engineered Geometric Features

This notebook evaluates the **Zonal / Edge_arc_feat** ablation for:
- `ArGEnT_self_att_noSDF`
- `PointNetMLPJoint_headfeat`
- `PointNetMLPJoint_FP_headfeat`

> ArGEnT consumes the engineered descriptors through its self-attended point-token input. Unlike the PointNet models, its checkpoint does not use the PointNet-specific `extra_feat_cols` metadata field; its training-script `INPUT_COLS` define the authoritative feature configuration.

Evaluation method: **validation-split evaluation** (deterministic 80/20 geometry split, seed 42). Full independence from checkpoint selection is not proved.


In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, inspect, json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists(): raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
RESULTS_DIR = COMPARISON_DIR / 'results' / '02_engineered_geometric_features'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh
SPLIT_SEED, EVAL_FRACTION = 42, 0.20
REGIME = 'Zonal'
ABLATION = 'Edge_arc_feat'
FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP_headfeat']
DATASET_PATH = REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5'
QUALITATIVE_EXAMPLE_PATH = COMPARISON_DIR / 'Examples' / 'disc_example_edge_deriv_zonal.h5'
FEATURE_LABELS = {
    0: 'x_mm',
    1: 'r_mm',
    2: 'zone_id',
    3: 'arc_length_mm',
    4: 'tangent_x',
    5: 'tangent_r',
    6: 'curvature',
    7: 'curvature_gradient',
}
COMMIT = __import__('subprocess').check_output(['git','rev-parse','HEAD'], cwd=REPO_ROOT, text=True).strip()
display(Markdown(
    f'**Inspected commit:** `{COMMIT}`\n\n'
    f'**Results directory:** `{RESULTS_DIR}`\n\n'
    f'**Evaluation label:** `validation-split evaluation`'
))


## Checkpoint discovery, script metadata, and ArGEnT feature-path audit inputs

Discovery validates checkpoint compatibility and parses each colocated training script. For ArGEnT, `Training_script.py` (`INPUT_COLS`, `QUERY_COLS`) is treated as authoritative for inference feature wiring.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def decode(value):
    return value.decode() if isinstance(value, bytes) else value

def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value

def compact_hash(obj):
    payload = json.dumps(json_ready(obj), sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()[:16]

def parse_script_metadata(script_path: Path) -> dict:
    meta = {'path': str(script_path)}
    tree = ast.parse(script_path.read_text(encoding='utf-8'))
    wanted = {
        'INPUT_COLS', 'QUERY_COLS', 'EXTRA_FEAT_COLS', 'HEAD_FEAT_COLS', 'TARGET_NAMES',
        'EXPECTED_REPR', 'H5_FILENAME', 'Perc_training_data', 'train_data_percent'
    }
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in wanted:
                    try:
                        meta[target.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in wanted:
            try:
                meta[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
        elif isinstance(node, ast.Call) and getattr(node.func, 'id', None) == 'train_test_split':
            for kw in node.keywords:
                if kw.arg in {'test_size', 'random_state'}:
                    try:
                        meta[kw.arg] = ast.literal_eval(kw.value)
                    except Exception:
                        pass
    return meta

def expected_encoder_cols(family: str, payload: dict, script_meta: dict) -> list[int]:
    if family.startswith('ArGEnT'):
        cols = script_meta.get('INPUT_COLS') or (payload.get('arch') or {}).get('input_cols') or [0, 1]
        return [int(c) for c in cols]
    cols = payload.get('extra_feat_cols') or script_meta.get('EXTRA_FEAT_COLS') or []
    return [int(c) for c in cols]

def expected_head_cols(family: str, payload: dict, script_meta: dict) -> list[int]:
    cols = payload.get('head_feat_cols', payload.get('headfeatcols'))
    if cols is None:
        cols = script_meta.get('HEAD_FEAT_COLS') or []
    return [int(c) for c in cols]

def discover_checkpoints():
    rows = []
    for family in FAMILIES:
        folder = REPO_ROOT / REGIME / ABLATION / family
        checkpoints = sorted((folder / 'Trained_models').glob('*.pt'))
        scripts = sorted(folder.glob('Training_script*.py'))
        script_meta = parse_script_metadata(scripts[0]) if scripts else {}
        if not checkpoints:
            rows.append({
                'regime': REGIME,
                'ablation': ABLATION,
                'model_family': family,
                'status': 'missing checkpoint',
                'checkpoint_path': None,
                'training_scripts': [str(p) for p in scripts],
            })
            continue
        for path in checkpoints:
            status = 'discovered'
            metadata = {}
            try:
                payload = torch.load(path, map_location='cpu', weights_only=False)
                required = ['arch', 'model_state', 'coord_center', 'coord_half_range', 'target_mean', 'target_std']
                missing = [k for k in required if k not in payload]
                if missing:
                    status = 'incompatible: missing ' + ', '.join(missing)
                if family.endswith('headfeat') and not (payload.get('head_feat_cols', payload.get('headfeatcols'))):
                    status = 'incompatible: missing head-feature metadata'
                metadata = {
                    'arch': json_ready(payload.get('arch')),
                    'arch_hash': compact_hash(payload.get('arch')) if payload.get('arch') is not None else None,
                    'model_name': payload.get('model_name'),
                    'target_names': json_ready(payload.get('target_names')),
                    'extra_feat_cols': json_ready(payload.get('extra_feat_cols')),
                    'head_feat_cols': json_ready(payload.get('head_feat_cols', payload.get('headfeatcols'))),
                    'encoder_input_cols': expected_encoder_cols(family, payload, script_meta),
                    'head_input_cols': expected_head_cols(family, payload, script_meta),
                    'script_meta': json_ready(script_meta),
                    'script_repr': script_meta.get('EXPECTED_REPR'),
                    'script_h5_filename': script_meta.get('H5_FILENAME'),
                }
                if script_meta.get('EXPECTED_REPR') not in (None, 'edge'):
                    status = f"incompatible: training script expects representation {script_meta.get('EXPECTED_REPR')!r}"
                if script_meta.get('H5_FILENAME') not in (None, DATASET_PATH.name):
                    status = f"incompatible: training script points to {script_meta.get('H5_FILENAME')!r}"
            except Exception as exc:
                status = 'incompatible: ' + type(exc).__name__ + ': ' + str(exc)
            rows.append({
                'regime': REGIME,
                'ablation': ABLATION,
                'model_family': family,
                'status': status,
                'checkpoint_path': str(path),
                'file_size_bytes': path.stat().st_size,
                'sha256': sha256(path),
                'training_scripts': [str(p) for p in scripts],
                **metadata,
            })
    report = pd.DataFrame(rows)
    report.to_json(RESULTS_DIR / 'checkpoint_integrity.json', orient='records', indent=2)
    return report

checkpoint_report = discover_checkpoints()
display(checkpoint_report[[
    'model_family', 'status', 'checkpoint_path', 'file_size_bytes', 'arch_hash',
    'extra_feat_cols', 'head_feat_cols'
]])


In [ ]:
def names_for_cols(cols):
    return [FEATURE_LABELS.get(int(c), f'feature_col_{c}') for c in (cols or [])]

feature_rows = []
for family in FAMILIES:
    sub = checkpoint_report[(checkpoint_report.model_family == family) & (checkpoint_report.status == 'discovered')]
    if sub.empty:
        feature_rows.append({
            'model_family': family,
            'encoder_full_columns': None,
            'encoder_inputs': None,
            'encoder_engineered_features': None,
            'head_columns': None,
            'head_inputs': None,
            'note': 'checkpoint unavailable',
        })
        continue
    row = sub.iloc[0]
    encoder_cols = row.encoder_input_cols or []
    head_cols = row.head_input_cols or []
    engineered_encoder = [c for c in encoder_cols if int(c) >= 3]
    note = 'Head uses explicit geometric features.' if head_cols else 'No explicit head features.'
    if family.startswith('ArGEnT'):
        note = 'Authoritative ArGEnT engineered-feature path is Training_script.py INPUT_COLS.'
    feature_rows.append({
        'model_family': family,
        'encoder_full_columns': encoder_cols,
        'encoder_inputs': ', '.join(names_for_cols(encoder_cols)),
        'encoder_engineered_features': ', '.join(names_for_cols(engineered_encoder)) or '(none)',
        'head_columns': head_cols,
        'head_inputs': ', '.join(names_for_cols(head_cols)) or '(x, r only)',
        'note': note,
    })
feature_assignment = pd.DataFrame(feature_rows)
eh.save_table(feature_assignment, RESULTS_DIR, 'feature_assignment')
display(feature_assignment)


## HDF5 loading and fixed evaluation split

Quantitative metrics in this notebook use only `Data_gen/output/disc_dataset_edge_deriv_zonal.h5` with the deterministic geometry-level validation split.


In [ ]:

def load_samples(path):
    samples = []
    if not path.exists():
        return samples
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge':
            raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        node_feature_names = [decode(x) for x in np.asarray(h5.get('node_feature_names', []))] if 'node_feature_names' in h5 else []
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None):
                return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm')
            stress = arr('stress_max_vm')
            life = arr('life_raw')
            if coords is None or stress is None or life is None:
                raise ValueError(f'{path.name}/{key}: missing required target fields')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            local_names = [decode(x) for x in np.asarray(g['node_feature_names'])] if 'node_feature_names' in g else node_feature_names
            arc_length = arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1).astype('float32')
            raw_node_features = arr('node_features', np.empty((len(coords), 0), dtype='float32')).astype('float32')
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1).astype('float32'),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arc_length,
                'raw_node_features': raw_node_features,
                'node_feature_names': local_names,
            })
    return samples


def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(samples))
    n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist()
    train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos

all_samples = load_samples(DATASET_PATH)
dataset_available = bool(all_samples)
train_pos, eval_pos = split_samples(all_samples) if dataset_available else ([], [])
split_record = {
    'regime': REGIME,
    'ablation': ABLATION,
    'dataset_path': str(DATASET_PATH),
    'dataset_available': bool(dataset_available),
    'total_geometry_count': len(all_samples),
    'training_geometry_count': len(train_pos),
    'evaluation_geometry_count': len(eval_pos),
    'training_sample_ids': [all_samples[i]['sample_id'] for i in train_pos],
    'evaluation_sample_ids': [all_samples[i]['sample_id'] for i in eval_pos],
    'split_seed': SPLIT_SEED,
    'split_fraction': EVAL_FRACTION,
    'evaluation_label': 'validation-split evaluation',
    'independence_basis': 'A deterministic geometry holdout is used, but checkpoint-selection independence is not proved.',
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'notebook_commit_sha': COMMIT,
}
with open(RESULTS_DIR / 'evaluation_split_provenance.json', 'w', encoding='utf-8') as stream:
    json.dump(split_record, stream, indent=2)
eh.save_json({
    'dataset_available': bool(dataset_available),
    'dataset_path': str(DATASET_PATH),
}, RESULTS_DIR, 'dataset_status')
display(pd.DataFrame([{
    'regime': REGIME,
    'ablation': ABLATION,
    'total_geometries': len(all_samples),
    'evaluation_geometries': len(eval_pos),
    'dataset_available': bool(dataset_available),
    'label': 'validation-split evaluation',
}]))


## Model reconstruction, ArGEnT preflight input audit, and shared-geometry inference


In [ ]:

def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot load module from {path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent
    family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False)
    arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f"pn_{folder.parent.name}_{folder.name}_{family}")
    sys.modules['pn_models'] = pn
    if family in ('PointNetMLPJoint_FP', 'PointNetMLPJoint_FP_headfeat'):
        if not hasattr(pn, 'build_fp_model_from_arch'):
            raise RuntimeError('FP checkpoint requires build_fp_model_from_arch')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted', 'PointNetMLPJoint_headfeat'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{family}')
        if not hasattr(bench, 'ArGEnTDeepONet'):
            raise RuntimeError('No ArGEnTDeepONet found in own benchmarks.py')
        defaults = {'hidden_dim': 128, 'num_heads': 4, 'num_layers': 2, 'output_dim': 128, 'out_channels': 1, 'attention_type': 'self', 'use_sdf': False, 'in_ch_geom': 2}
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']:
            defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt


def _get_stats_entry(stats, col):
    if isinstance(stats, dict):
        return stats.get(col, stats.get(str(col)))
    return None


def normalization_vectors(ckpt, cols):
    cols = [int(c) for c in (cols or [])]
    stats = ckpt.get('extra_feat_stats')
    if cols and stats is None:
        raise ValueError('missing extra_feat_stats')
    means, stds = [], []
    for i, col in enumerate(cols):
        entry = _get_stats_entry(stats, int(col))
        if entry is not None:
            if isinstance(entry, dict):
                means.append(float(entry.get('mean', 0.0)))
                stds.append(float(entry.get('std', 1.0)))
            else:
                means.append(float(entry[0]))
                stds.append(float(entry[1]))
        elif isinstance(stats, (list, tuple, np.ndarray)):
            means.append(float(stats[0][i]))
            stds.append(float(stats[1][i]))
        else:
            raise ValueError(f'missing normalization entry for feature column {col}')
    return np.asarray(means, dtype='float32'), np.maximum(np.asarray(stds, dtype='float32'), 1e-8)


def source_column_names(sample):
    raw_names = list(sample.get('node_feature_names') or [])
    raw_width = sample['raw_node_features'].shape[1] if sample['raw_node_features'].ndim == 2 else 0
    if len(raw_names) < raw_width:
        raw_names = raw_names + [f'raw_feature_{i}' for i in range(len(raw_names), raw_width)]
    return ['x_mm', 'r_mm', 'zone_id', 'arc_length_mm', *raw_names[:raw_width]]


def source_column_matrix(sample):
    cols = [
        sample['coords'][:, 0],
        sample['coords'][:, 1],
        sample['zone_id'],
        sample['arc_length_mm'],
    ]
    nf = sample['raw_node_features']
    if nf.ndim == 1:
        nf = nf[:, None]
    for j in range(nf.shape[1]):
        cols.append(nf[:, j])
    return np.column_stack(cols).astype('float32')


def resolve_feature_matrix(sample, requested_cols, label):
    requested_cols = [int(c) for c in (requested_cols or [])]
    source = source_column_matrix(sample)
    names = source_column_names(sample)
    if requested_cols:
        if source.shape[1] <= max(requested_cols):
            raise ValueError(f'{label}: source width {source.shape[1]} too small for requested columns {requested_cols}')
        data = source[:, requested_cols].astype('float32')
        feat_names = [names[c] if c < len(names) else f'feature_col_{c}' for c in requested_cols]
    else:
        data = np.empty((source.shape[0], 0), dtype='float32')
        feat_names = []
    return source, data, feat_names, names


def normalize_argent_tensor(source, input_cols, ckpt):
    center = np.asarray(ckpt['coord_center'], dtype='float32').reshape(-1)
    half = np.maximum(np.asarray(ckpt['coord_half_range'], dtype='float32').reshape(-1), 1e-8)
    stats = ckpt.get('extra_feat_stats')
    out = np.empty((source.shape[0], len(input_cols)), dtype='float32')
    for j, c in enumerate(input_cols):
        c = int(c)
        vec = source[:, c].astype('float32')
        if c == 0:
            out[:, j] = (vec - center[0]) / half[0]
        elif c == 1:
            out[:, j] = (vec - center[1]) / half[1]
        else:
            entry = _get_stats_entry(stats, c)
            if entry is None:
                raise ValueError(f'missing extra_feat_stats entry for INPUT_COLS column {c}')
            if isinstance(entry, dict):
                mean = float(entry.get('mean', 0.0)); std = max(float(entry.get('std', 1.0)), 1e-8)
            else:
                mean = float(entry[0]); std = max(float(entry[1]), 1e-8)
            out[:, j] = (vec - mean) / std
    return out


def normalize_query_from_cols(source, query_cols, ckpt):
    center = np.asarray(ckpt['coord_center'], dtype='float32').reshape(-1)
    half = np.maximum(np.asarray(ckpt['coord_half_range'], dtype='float32').reshape(-1), 1e-8)
    q = np.empty((source.shape[0], len(query_cols)), dtype='float32')
    for j, c in enumerate(query_cols):
        c = int(c)
        if c == 0:
            q[:, j] = (source[:, c] - center[0]) / half[0]
        elif c == 1:
            q[:, j] = (source[:, c] - center[1]) / half[1]
        else:
            raise ValueError(f'Unexpected QUERY_COLS entry {c}; notebook expects x/r query columns')
    return q


def decode_prediction(out, ckpt):
    out_np = out.detach().cpu().numpy()
    if out_np.ndim != 3 or out_np.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out_np.shape}')
    mean = np.asarray(ckpt['target_mean'], dtype='float32')
    std = np.asarray(ckpt['target_std'], dtype='float32')
    out_np = out_np * std + mean
    if out_np.shape[2] == 2:
        return out_np[0, :, 0], out_np[0, :, 1], tuple(out.shape)
    if out_np.shape[2] == 1:
        return np.zeros(out_np.shape[1], dtype='float32'), out_np[0, :, 0], tuple(out.shape)
    raise ValueError(f'unexpected output channels: {out_np.shape[2]}')


def compact_feature_stats(x: np.ndarray):
    if x.size == 0:
        return []
    rows = []
    for j in range(x.shape[1]):
        col = x[:, j]
        rows.append({
            'min': float(np.nanmin(col)),
            'mean': float(np.nanmean(col)),
            'max': float(np.nanmax(col)),
        })
    return rows


def make_feature_audit(row, ckpt, expected_names, actual_names, tensor, preproc_path, n_eval_samples, prediction_rows, baseline_match=None, status='ok'):
    return {
        'checkpoint_identifier': row['checkpoint_path'],
        'model_family': row['model_family'],
        'expected_feature_count': len(expected_names),
        'actual_feature_count': int(tensor.shape[1]) if tensor is not None else 0,
        'training_feature_names': expected_names,
        'inference_feature_names': actual_names,
        'feature_tensor_shape': list(tensor.shape) if tensor is not None else None,
        'dtype': str(tensor.dtype) if tensor is not None else None,
        'feature_stats': compact_feature_stats(tensor) if tensor is not None else [],
        'nan_count': int(np.isnan(tensor).sum()) if tensor is not None else 0,
        'inf_count': int(np.isinf(tensor).sum()) if tensor is not None else 0,
        'nan_fraction': float(np.isnan(tensor).mean()) if tensor is not None and tensor.size else 0.0,
        'inf_fraction': float(np.isinf(tensor).mean()) if tensor is not None and tensor.size else 0.0,
        'preprocessing_artifact_identifier': preproc_path,
        'number_of_evaluated_samples': int(n_eval_samples),
        'number_of_prediction_rows': int(prediction_rows),
        'prediction_sample_ids_exactly_match_baseline': baseline_match,
        'status': status,
    }


def predict_argent(model, sample, ckpt, script_meta, checkpoint_path):
    input_cols = [int(c) for c in (script_meta.get('INPUT_COLS') or [])]
    query_cols = [int(c) for c in (script_meta.get('QUERY_COLS') or [0, 1])]
    if not input_cols:
        raise ValueError('ArGEnT script metadata missing INPUT_COLS')
    source, _, _, source_names = resolve_feature_matrix(sample, input_cols, 'ArGEnT source')
    if source.shape[1] <= max(input_cols):
        raise ValueError(f'source tensor width {source.shape[1]} too small for INPUT_COLS={input_cols}')
    expected_feature_names = [source_names[c] for c in input_cols]
    geom = normalize_argent_tensor(source, input_cols, ckpt)
    q = normalize_query_from_cols(source, query_cols, ckpt)
    assert geom.shape[-1] == len(input_cols)
    assert np.isfinite(geom).all()
    x_t = torch.from_numpy(geom[None])
    q_t = torch.from_numpy(q[None])
    kwargs = {}
    sig = inspect.signature(model.forward)
    if 'mask' in sig.parameters:
        kwargs['mask'] = torch.ones((1, q_t.shape[1]), dtype=torch.bool)
    if 'kv_mask' in sig.parameters:
        kwargs['kv_mask'] = torch.ones((1, x_t.shape[1]), dtype=torch.bool)
    with torch.no_grad():
        out = model(x_t, q_t, **kwargs)
    pred_stress, pred_loglife, out_shape = decode_prediction(out, ckpt)
    arch_in = int((ckpt.get('arch') or {}).get('in_ch_geom', -1))
    if arch_in > 0 and int(x_t.shape[-1]) != arch_in:
        raise ValueError(f'ArGEnT in_ch_geom mismatch: inferred {int(x_t.shape[-1])} vs checkpoint arch {arch_in}')
    if out.shape[-1] != 2:
        raise ValueError(f'ArGEnT forward output must be [B,Q,2], got {tuple(out.shape)}')
    audit = {
        'expected_feature_names': expected_feature_names,
        'actual_feature_names': expected_feature_names,
        'tensor': geom,
        'out_shape': list(out_shape),
        'preprocessing_artifact_identifier': f"{checkpoint_path}::extra_feat_stats/coord_center/coord_half_range",
    }
    return pred_stress, pred_loglife, audit


def predict_headfeat(model, sample, ckpt, family, checkpoint_path):
    extra_feat_cols = [int(c) for c in (ckpt.get('extra_feat_cols', []) or [])]
    head_feat_cols = [int(c) for c in (ckpt.get('head_feat_cols', ckpt.get('headfeatcols')) or [])]
    source, geom_raw, geom_names, source_names = resolve_feature_matrix(sample, extra_feat_cols, 'encoder engineered features')
    _, head_raw, head_names, _ = resolve_feature_matrix(sample, head_feat_cols, 'head engineered features')
    assert geom_names == [source_names[c] for c in extra_feat_cols]
    assert head_names == [source_names[c] for c in head_feat_cols]
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    coords_norm = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if geom_raw.shape[1] > 0:
        means, stds = normalization_vectors(ckpt, extra_feat_cols)
        geom_norm = (geom_raw - means) / stds
    else:
        geom_norm = np.empty((len(sample['coords']), 0), dtype='float32')
    if head_raw.shape[1] > 0:
        means_h, stds_h = normalization_vectors(ckpt, head_feat_cols)
        head_norm = (head_raw - means_h) / stds_h
    else:
        head_norm = np.empty((len(sample['coords']), 0), dtype='float32')
    assert geom_norm.shape[-1] == len(extra_feat_cols)
    assert head_norm.shape[-1] == len(head_feat_cols)
    assert np.isfinite(geom_norm).all() and np.isfinite(head_norm).all()
    x = torch.from_numpy(coords_norm.astype('float32')[None])
    q = x.clone()
    gf = torch.from_numpy(geom_norm[None]) if geom_norm.shape[1] > 0 else None
    hf = torch.from_numpy(head_norm[None]) if head_norm.shape[1] > 0 else None
    with torch.no_grad():
        if family == 'PointNetMLPJoint_FP_headfeat':
            out = model(x, q, geom_feats=gf, headfeats=hf)
        elif family == 'PointNetMLPJoint_headfeat':
            out = model(x, q, geom_feats=gf, head_feats=hf)
        else:
            raise ValueError(f'Unknown headfeat family: {family}')
    pred_stress, pred_loglife, _ = decode_prediction(out, ckpt)
    audit = {
        'expected_feature_names': geom_names,
        'actual_feature_names': geom_names,
        'tensor': geom_norm,
        'head_feature_names': head_names,
        'head_tensor': head_norm,
        'preprocessing_artifact_identifier': f"{checkpoint_path}::extra_feat_stats/coord_center/coord_half_range",
    }
    return pred_stress, pred_loglife, audit


def predict_dispatch(model, sample, ckpt, family, script_meta):
    if family == 'ArGEnT_self_att_noSDF':
        return predict_argent(model, sample, ckpt, script_meta, ckpt.get('checkpoint_path', 'checkpoint'))
    if family in ('PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP_headfeat'):
        extra_feat_cols = [int(c) for c in (ckpt.get('extra_feat_cols', []) or [])]
        head_feat_cols = [int(c) for c in (ckpt.get('head_feat_cols', ckpt.get('headfeatcols')) or [])]
        if extra_feat_cols != [3, 4, 5, 6, 7] or head_feat_cols != [3, 4, 5, 6, 7]:
            raise ValueError(f'unexpected headfeat feature configuration: extra={extra_feat_cols}, head={head_feat_cols}')
        return predict_headfeat(model, sample, ckpt, family, ckpt.get('checkpoint_path', 'checkpoint'))
    raise ValueError(f'Unexpected family in notebook 02: {family}')


node_frames, load_errors, coverage_rows = [], [], []
feature_audits = []

if dataset_available:
    for _, row in checkpoint_report.iterrows():
        if row.status != 'discovered':
            continue
        try:
            model, ckpt = reconstruct(row)
            ckpt['checkpoint_path'] = row.checkpoint_path
            script_meta = row.script_meta if isinstance(row.script_meta, dict) else {}
            expected = ckpt.get('target_names', ['Stress', 'LogLife'])
            if len(expected) != 2:
                raise ValueError(f'incompatible target dimensions: {expected}')
            predicted_ids = []
            first_audit = None
            for i in eval_pos:
                s = all_samples[i]
                pred_stress, pred_loglife, audit = predict_dispatch(model, s, ckpt, row.model_family, script_meta)
                if first_audit is None:
                    first_audit = audit
                predicted_ids.append(s['sample_id'])
                base = pd.DataFrame({
                    'regime': REGIME,
                    'ablation': ABLATION,
                    'model_family': row.model_family,
                    'sample_key': s['sample_key'],
                    'sample_id': s['sample_id'],
                    'node_idx': np.arange(len(s['coords'])),
                    'x_mm': s['coords'][:, 0],
                    'r_mm': s['coords'][:, 1],
                    'zone_id': s['zone_id'],
                    'subzone_id': s['subzone_id'],
                    'arc_length_mm': s['arc_length_mm'],
                    'true_stress': s['stress'],
                    'pred_stress': pred_stress,
                    'true_loglife': s['loglife'],
                    'pred_loglife': pred_loglife,
                })
                base['zone_name'] = base.zone_id.map(eh.ZONE_ID_TO_NAME)
                base['subzone_name'] = base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
                node_frames.append(base)
            coverage_rows.append({'model_family': row.model_family, 'successful_eval_geometries': len(predicted_ids), 'sample_ids': predicted_ids})
            if first_audit is not None:
                feature_audits.append(make_feature_audit(
                    row, ckpt,
                    expected_names=first_audit.get('expected_feature_names', []),
                    actual_names=first_audit.get('actual_feature_names', []),
                    tensor=first_audit.get('tensor'),
                    preproc_path=first_audit.get('preprocessing_artifact_identifier'),
                    n_eval_samples=len(predicted_ids),
                    prediction_rows=sum(len(all_samples[i]['coords']) for i in eval_pos),
                    baseline_match=None,
                    status='ok',
                ))
        except Exception as exc:
            load_errors.append({'model_family': row.model_family, 'checkpoint_path': row.checkpoint_path, 'status': 'load/inference failed: ' + repr(exc)})
else:
    for _, row in checkpoint_report.iterrows():
        if row.status == 'discovered':
            script_meta = row.script_meta if isinstance(row.script_meta, dict) else {}
            expected_names = names_for_cols(row.encoder_input_cols if row.model_family.startswith('ArGEnT') else row.head_input_cols)
            feature_audits.append(make_feature_audit(
                row, {'arch': row.arch if isinstance(row.arch, dict) else {}},
                expected_names=expected_names,
                actual_names=expected_names,
                tensor=None,
                preproc_path=f"{row.checkpoint_path}::extra_feat_stats/coord_center/coord_half_range",
                n_eval_samples=0,
                prediction_rows=0,
                baseline_match=None,
                status='dataset unavailable; static metadata-only audit',
            ))

nodes = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame()
pd.DataFrame(load_errors).to_json(RESULTS_DIR / 'inference_errors.json', orient='records', indent=2)
coverage = pd.DataFrame(coverage_rows)
if not coverage.empty:
    shared_ids = sorted(set.intersection(*[set(ids) for ids in coverage['sample_ids']]))
else:
    shared_ids = []
coverage_summary = pd.DataFrame([
    {
        'model_family': fam,
        'successful_eval_geometries': len(ids),
        'shared_eval_geometries': len(shared_ids),
        'dropped_to_enforce_fairness': len(ids) - len(shared_ids),
    }
    for fam, ids in zip(coverage['model_family'], coverage['sample_ids'])
]) if not coverage.empty else pd.DataFrame(columns=['model_family', 'successful_eval_geometries', 'shared_eval_geometries', 'dropped_to_enforce_fairness'])
coverage_summary['evaluation_label'] = 'validation-split evaluation' if not coverage_summary.empty else []
eh.save_table(coverage_summary, RESULTS_DIR, 'evaluation_geometry_coverage')
if shared_ids:
    nodes = nodes[nodes.sample_id.isin(shared_ids)].copy()
feature_validation_df = pd.DataFrame(feature_audits)
eh.save_table(feature_validation_df if not feature_validation_df.empty else pd.DataFrame(columns=[
    'checkpoint_identifier', 'model_family', 'expected_feature_count', 'actual_feature_count',
    'training_feature_names', 'inference_feature_names', 'feature_tensor_shape', 'dtype',
    'feature_stats', 'nan_count', 'inf_count', 'nan_fraction', 'inf_fraction',
    'preprocessing_artifact_identifier', 'number_of_evaluated_samples', 'number_of_prediction_rows',
    'prediction_sample_ids_exactly_match_baseline', 'status'
]), RESULTS_DIR, 'feature_inference_validation')
display(pd.DataFrame(load_errors) if load_errors else coverage_summary)
display(feature_validation_df)


## Compact metrics and figures


In [ ]:

pooled = eh.pooled_metrics_from_nodes(nodes) if not nodes.empty else pd.DataFrame()
life_bands = eh.loglife_bin_metrics(nodes) if not nodes.empty else pd.DataFrame(columns=['model_family', 'life_bin', 'n_samples', 'mae_loglife', 'rmse_loglife'])
grouped_regions = eh.grouped_region_metrics_from_nodes(nodes) if not nodes.empty else pd.DataFrame()
geom = eh.geometry_level_metrics(nodes) if not nodes.empty else pd.DataFrame()

summary_rows = []
for family in FAMILIES:
    row = {'evaluation_label': 'validation-split evaluation', 'regime': REGIME, 'ablation': ABLATION, 'model_family': family}
    for target in ('Stress', 'LogLife'):
        sub = pooled[(pooled['model_family'] == family) & (pooled['target'] == target)] if not pooled.empty else pd.DataFrame()
        if not sub.empty:
            row[f'{target}_MAE'] = float(sub.iloc[0]['MAE'])
            row[f'{target}_RMSE'] = float(sub.iloc[0]['RMSE'])
            row[f'{target}_R2'] = float(sub.iloc[0]['R2 (log)'])
    full_sub = life_bands[(life_bands['model_family'] == family) & (life_bands['life_bin'] == eh.FULL_TEST_SET_LABEL)] if not life_bands.empty else pd.DataFrame()
    row['n_samples_full_test'] = int(full_sub.iloc[0]['n_samples']) if not full_sub.empty else 0
    gsub = geom[geom['model_family'] == family] if not geom.empty else pd.DataFrame()
    if not gsub.empty:
        row['whole_geometry_mean_LogLife_MAE'] = float(gsub['whole_geometry_loglife_mae'].mean())
        row['whole_geometry_mean_Stress_MAE'] = float(gsub['whole_geometry_stress_mae'].mean())
        row['mean_abs_min_LogLife_error'] = float(gsub['abs_min_loglife_error_decades'].mean())
        row['mean_abs_max_Stress_error'] = float(gsub['abs_max_stress_error'].mean())
        row['critical_zone_agreement'] = float(gsub['same_zone_critical'].mean())
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
geometry_summary = geom.groupby(['regime', 'ablation', 'model_family'], as_index=False).agg(
    n_geometries=('sample_id', 'nunique'),
    whole_geometry_mean_loglife_mae=('whole_geometry_loglife_mae', 'mean'),
    whole_geometry_mean_stress_mae=('whole_geometry_stress_mae', 'mean'),
    absolute_min_loglife_error_mean=('abs_min_loglife_error_decades', 'mean'),
    absolute_max_stress_error_mean=('abs_max_stress_error', 'mean'),
    critical_zone_agreement=('same_zone_critical', 'mean'),
) if not geom.empty else pd.DataFrame(columns=['regime', 'ablation', 'model_family', 'n_geometries'])

pairs = []
for left, right in [('ArGEnT_self_att_noSDF', 'PointNetMLPJoint_headfeat'), ('ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP_headfeat'), ('PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP_headfeat')]:
    if geom.empty:
        continue
    l = geom[geom['model_family'] == left].set_index('sample_id')
    r = geom[geom['model_family'] == right].set_index('sample_id')
    common = sorted(set(l.index).intersection(r.index))
    if not common:
        continue
    d = pd.Series([float(l.loc[sid]['abs_min_loglife_error_decades'] - r.loc[sid]['abs_min_loglife_error_decades']) for sid in common])
    pairs.append({
        'left_family': left,
        'right_family': right,
        'n_geometries': len(common),
        'median_left_minus_right_abs_min_loglife_err': float(d.median()),
        'fraction_right_better': float((d > 0).mean()),
    })
paired_summary = pd.DataFrame(pairs)

for name, frame in [
    ('summary_table', summary_table),
    ('life_band_metrics', life_bands),
    ('grouped_region_metrics', grouped_regions if not grouped_regions.empty else pd.DataFrame(columns=['regime', 'ablation', 'model_family', 'grouped_region', 'n_nodes', 'LogLife_MAE', 'LogLife_RMSE', 'Stress_MAE', 'Stress_RMSE', 'status'])),
    ('geometry_summary', geometry_summary),
    ('paired_summary', paired_summary if not paired_summary.empty else pd.DataFrame(columns=['left_family', 'right_family', 'n_geometries', 'median_left_minus_right_abs_min_loglife_err', 'fraction_right_better'])),
]:
    eh.save_table(frame, RESULTS_DIR, name)

if not pooled.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, target in zip(axes, ['Stress', 'LogLife']):
        sub = pooled[pooled['target'] == target].set_index('model_family').reindex(FAMILIES)
        ax.bar(np.arange(len(FAMILIES)), sub['MAE'])
        ax.set_xticks(np.arange(len(FAMILIES)))
        ax.set_xticklabels(FAMILIES, rotation=20, ha='right')
        ax.set_ylabel(f'{target} MAE')
        ax.set_title(f'{target} pooled MAE')
    fig.tight_layout(); fig.savefig(FIGURES_DIR / 'figure_main_metrics.png', dpi=150, bbox_inches='tight'); fig.savefig(FIGURES_DIR / 'figure_main_metrics.pdf', bbox_inches='tight'); plt.close(fig)

if not life_bands.empty:
    eh.plot_bin_bar(life_bands, f'{REGIME} {ABLATION} life-band log-life errors', out_dir=FIGURES_DIR, filename='figure_life_bands')
    plt.close('all')

if not grouped_regions.empty:
    gr = grouped_regions[grouped_regions['status'] != 'missing'].copy()
    if not gr.empty:
        fig, ax = plt.subplots(figsize=(10, 4.5))
        regions = list(eh.GROUPED_REGION_DEFS.keys())
        x = np.arange(len(regions), dtype=float)
        width = 0.8 / max(1, len(FAMILIES))
        for i, fam in enumerate(FAMILIES):
            sub = gr[gr['model_family'] == fam].set_index('grouped_region')
            vals = [sub.loc[r, 'LogLife_MAE'] if r in sub.index else np.nan for r in regions]
            ax.bar(x + (i - (len(FAMILIES)-1)/2)*width, vals, width=width, label=fam)
        ax.set_xticks(x); ax.set_xticklabels(regions, rotation=20, ha='right')
        ax.set_ylabel('log_life MAE (decades)'); ax.set_title('Grouped physical regions')
        ax.legend(fontsize=7); fig.tight_layout()
        fig.savefig(FIGURES_DIR / 'figure_grouped_regions.png', dpi=150, bbox_inches='tight')
        fig.savefig(FIGURES_DIR / 'figure_grouped_regions.pdf', bbox_inches='tight')
        plt.close(fig)

if dataset_available and QUALITATIVE_EXAMPLE_PATH.exists():
    ex_samples = load_samples(QUALITATIVE_EXAMPLE_PATH)
    if ex_samples:
        ex = ex_samples[0]
        by_model = {}
        for _, row in checkpoint_report[checkpoint_report.status == 'discovered'].iterrows():
            model, ckpt = reconstruct(row)
            script_meta = row.script_meta if isinstance(row.script_meta, dict) else {}
            ps, pl, _ = predict_dispatch(model, ex, ckpt, row.model_family, script_meta)
            f = pd.DataFrame({
                'x_mm': ex['coords'][:, 0], 'r_mm': ex['coords'][:, 1],
                'true_loglife': ex['loglife'], 'pred_loglife': pl,
                'true_stress': ex['stress'], 'pred_stress': ps,
            })
            by_model[row.model_family] = f
        eh.plot_field_comparison(by_model, 'true_loglife', 'pred_loglife', 'decades', 'Qualitative illustrative example generated from the same FEM/data-generation pipeline; quantitative comparisons use the geometry-level validation split.', out_dir=FIGURES_DIR, filename='figure_qualitative_example')
        plt.close('all')

run_metadata = {
    'commit_sha': COMMIT,
    'evaluation_label': 'validation-split evaluation',
    'results_dir': str(RESULTS_DIR),
    'families': FAMILIES,
    'argent_feature_authority': 'Training_script.py INPUT_COLS/QUERY_COLS',
    'quantitative_dataset': str(DATASET_PATH),
    'qualitative_example_dataset': str(QUALITATIVE_EXAMPLE_PATH),
    'dataset_available': bool(dataset_available),
}
eh.save_json(run_metadata, RESULTS_DIR, 'run_metadata')

display(summary_table)
display(paired_summary)


## Notebook summary


In [ ]:

# ── Head-to-head: headfeat models vs regular baselines ──────────────────────
# delta = ablation - baseline; negative means the engineered-feature variant has lower error.

BASELINE_ABLATION = 'Edge'
ABLATION_TO_BASELINE_FAMILY = {
    'ArGEnT_self_att_noSDF': 'ArGEnT_self_att_noSDF',
    'PointNetMLPJoint_headfeat': 'PointNetMLPJoint',
    'PointNetMLPJoint_FP_headfeat': 'PointNetMLPJoint_FP',
}
BASELINE_FAMILIES_NEEDED = sorted(set(ABLATION_TO_BASELINE_FAMILY.values()))


def predict_baseline_pointnet(model, sample, ckpt):
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.maximum(np.asarray(ckpt['coord_half_range'], dtype='float32'), 1e-8)
    coords_norm = (sample['coords'] - center) / half
    x = torch.from_numpy(coords_norm[None])
    q = x.clone()
    with torch.no_grad():
        out = model(x, q)
    pred_stress, pred_loglife, _ = decode_prediction(out, ckpt)
    return pred_stress, pred_loglife

baseline_rows = []
for family in BASELINE_FAMILIES_NEEDED:
    folder = REPO_ROOT / REGIME / BASELINE_ABLATION / family
    checkpoints = sorted((folder / 'Trained_models').glob('*.pt')) if folder.exists() else []
    scripts = sorted(folder.glob('Training_script*.py')) if folder.exists() else []
    script_meta = parse_script_metadata(scripts[0]) if scripts else {}
    if not checkpoints:
        baseline_rows.append({'regime': REGIME, 'ablation': BASELINE_ABLATION, 'model_family': family, 'status': 'missing checkpoint', 'checkpoint_path': None, 'script_meta': {}})
        continue
    for path in checkpoints:
        try:
            payload = torch.load(path, map_location='cpu', weights_only=False)
            required = ['arch', 'model_state', 'coord_center', 'coord_half_range', 'target_mean', 'target_std']
            missing = [k for k in required if k not in payload]
            status = 'discovered' if not missing else f'incompatible: missing {missing}'
            baseline_rows.append({
                'regime': REGIME, 'ablation': BASELINE_ABLATION,
                'model_family': family, 'status': status,
                'checkpoint_path': str(path), 'sha256': sha256(path),
                'script_meta': script_meta, 'best_val_loss': payload.get('best_val_loss'),
            })
        except Exception as exc:
            baseline_rows.append({'regime': REGIME, 'ablation': BASELINE_ABLATION, 'model_family': family, 'status': f'skipped: {exc}', 'checkpoint_path': str(path), 'script_meta': {}})

baseline_checkpoint_report = pd.DataFrame(baseline_rows)
eh.save_table(baseline_checkpoint_report[['regime', 'ablation', 'model_family', 'status', 'checkpoint_path']].fillna(''), RESULTS_DIR, 'baseline_checkpoint_report')
display(baseline_checkpoint_report[['model_family', 'status', 'checkpoint_path']])

baseline_node_frames = []
baseline_load_errors = []
shared_set = set(shared_ids)
eval_samples_shared = [all_samples[i] for i in eval_pos if all_samples[i]['sample_id'] in shared_set] if dataset_available else []

for _, row in baseline_checkpoint_report[baseline_checkpoint_report['status'] == 'discovered'].iterrows():
    if not dataset_available:
        break
    try:
        model, ckpt = reconstruct(row)
        script_meta = row['script_meta'] if isinstance(row['script_meta'], dict) else {}
        for s in eval_samples_shared:
            family = row['model_family']
            if family == 'ArGEnT_self_att_noSDF':
                ps, pl, _ = predict_argent(model, s, ckpt, script_meta, row['checkpoint_path'])
            else:
                ps, pl = predict_baseline_pointnet(model, s, ckpt)
            base = pd.DataFrame({
                'regime': REGIME, 'ablation': BASELINE_ABLATION,
                'model_family': family,
                'sample_id': s['sample_id'],
                'node_idx': np.arange(len(s['coords'])),
                'x_mm': s['coords'][:, 0], 'r_mm': s['coords'][:, 1],
                'zone_id': s['zone_id'], 'subzone_id': s['subzone_id'],
                'arc_length_mm': s['arc_length_mm'],
                'true_stress': s['stress'], 'pred_stress': ps,
                'true_loglife': s['loglife'], 'pred_loglife': pl,
            })
            base['zone_name'] = base.zone_id.map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
            baseline_node_frames.append(base)
    except Exception as exc:
        baseline_load_errors.append({'model_family': row['model_family'], 'status': repr(exc)})

baseline_nodes = pd.concat(baseline_node_frames, ignore_index=True) if baseline_node_frames else pd.DataFrame()
baseline_pooled = eh.pooled_metrics_from_nodes(baseline_nodes) if not baseline_nodes.empty else pd.DataFrame()
baseline_life_bands = eh.loglife_bin_metrics(baseline_nodes) if not baseline_nodes.empty else pd.DataFrame(columns=['model_family', 'life_bin', 'n_samples', 'mae_loglife', 'rmse_loglife'])

paired_headfeat_rows = []
for ablation_fam, baseline_fam in ABLATION_TO_BASELINE_FAMILY.items():
    abl = life_bands[life_bands['model_family'] == ablation_fam] if not life_bands.empty else pd.DataFrame()
    base = baseline_life_bands[baseline_life_bands['model_family'] == baseline_fam] if not baseline_life_bands.empty else pd.DataFrame()
    if abl.empty or base.empty:
        continue
    merged = abl.merge(base, on='life_bin', suffixes=('_ablation', '_baseline'))
    for _, r in merged.iterrows():
        paired_headfeat_rows.append({
            'regime': REGIME,
            'model_family_baseline': baseline_fam,
            'model_family_ablation': ablation_fam,
            'baseline_ablation': BASELINE_ABLATION,
            'ablation': ABLATION,
            'evaluation_label': 'validation-split evaluation',
            'life_bin': r['life_bin'],
            'n_samples': int(r['n_samples_ablation']) if not pd.isna(r['n_samples_ablation']) else int(r['n_samples_baseline']),
            'n_shared_geometries': len(shared_ids),
            'baseline_mae_loglife': float(r['mae_loglife_baseline']) if not pd.isna(r['mae_loglife_baseline']) else np.nan,
            'ablation_mae_loglife': float(r['mae_loglife_ablation']) if not pd.isna(r['mae_loglife_ablation']) else np.nan,
            'delta_mae_loglife': float(r['mae_loglife_ablation'] - r['mae_loglife_baseline']) if not (pd.isna(r['mae_loglife_ablation']) or pd.isna(r['mae_loglife_baseline'])) else np.nan,
            'baseline_rmse_loglife': float(r['rmse_loglife_baseline']) if not pd.isna(r['rmse_loglife_baseline']) else np.nan,
            'ablation_rmse_loglife': float(r['rmse_loglife_ablation']) if not pd.isna(r['rmse_loglife_ablation']) else np.nan,
            'delta_rmse_loglife': float(r['rmse_loglife_ablation'] - r['rmse_loglife_baseline']) if not (pd.isna(r['rmse_loglife_ablation']) or pd.isna(r['rmse_loglife_baseline'])) else np.nan,
            'note_delta_sign': 'delta < 0 means the engineered-feature variant has lower error than the baseline',
        })

paired_headfeat_comparison = pd.DataFrame(paired_headfeat_rows)
fullset_pairs = paired_headfeat_comparison[paired_headfeat_comparison['life_bin'] == eh.FULL_TEST_SET_LABEL].copy() if not paired_headfeat_comparison.empty else pd.DataFrame()
eh.save_table(paired_headfeat_comparison if not paired_headfeat_comparison.empty else pd.DataFrame(columns=[
    'regime', 'model_family_baseline', 'model_family_ablation', 'baseline_ablation', 'ablation',
    'evaluation_label', 'life_bin', 'n_samples', 'n_shared_geometries', 'baseline_mae_loglife',
    'ablation_mae_loglife', 'delta_mae_loglife', 'baseline_rmse_loglife', 'ablation_rmse_loglife',
    'delta_rmse_loglife', 'note_delta_sign'
]), RESULTS_DIR, 'paired_headfeat_vs_baseline_by_life_bin')
eh.save_table(fullset_pairs if not fullset_pairs.empty else pd.DataFrame(columns=[
    'regime', 'model_family_baseline', 'model_family_ablation', 'baseline_ablation', 'ablation',
    'evaluation_label', 'life_bin', 'n_samples', 'n_shared_geometries', 'baseline_mae_loglife',
    'ablation_mae_loglife', 'delta_mae_loglife', 'baseline_rmse_loglife', 'ablation_rmse_loglife',
    'delta_rmse_loglife', 'note_delta_sign'
]), RESULTS_DIR, 'paired_headfeat_vs_baseline')

if not feature_validation_df.empty:
    sample_match = {}
    for fam in FAMILIES:
        if nodes.empty or baseline_nodes.empty:
            sample_match[fam] = None
        else:
            base_fam = ABLATION_TO_BASELINE_FAMILY.get(fam)
            left = sorted(nodes.loc[nodes['model_family'] == fam, 'sample_id'].unique().tolist())
            right = sorted(baseline_nodes.loc[baseline_nodes['model_family'] == base_fam, 'sample_id'].unique().tolist())
            sample_match[fam] = (left == right)
    feature_validation_df['prediction_sample_ids_exactly_match_baseline'] = feature_validation_df['model_family'].map(sample_match)
    eh.save_table(feature_validation_df, RESULTS_DIR, 'feature_inference_validation')

if not fullset_pairs.empty:
    pair_labels = [f"{r['model_family_ablation']}{chr(10)}vs{chr(10)}{r['model_family_baseline']}" for _, r in fullset_pairs.iterrows()]
    x_pos = np.arange(len(fullset_pairs))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, col, ylabel in zip(
        axes,
        ['delta_mae_loglife', 'delta_rmse_loglife'],
        ['ΔMAE, log₁₀(life) [decades]', 'ΔRMSE, log₁₀(life) [decades]'],
    ):
        bar_colors = ['#2ca02c' if v < 0 else '#d62728' for v in fullset_pairs[col]]
        ax.bar(x_pos, fullset_pairs[col], color=bar_colors, width=0.5, edgecolor='k', linewidth=0.6)
        ax.axhline(0, color='k', lw=1.2, zorder=5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(pair_labels, fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{col.replace('_', ' ').upper()}{chr(10)}(headfeat − baseline){chr(10)}Green = headfeat improves")
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'Engineered-feature ablation: full-test-set deltas{chr(10)}{REGIME} / {ABLATION} vs {REGIME} / {BASELINE_ABLATION}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'figure_headfeat_vs_baseline_delta.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURES_DIR / 'figure_headfeat_vs_baseline_delta.pdf', bbox_inches='tight')
    plt.close(fig)

if not paired_headfeat_comparison.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    life_bin_order = [label for label, _, _ in eh.ALL_LIFE_BIN_DEFS]
    for ax, delta_col, title in [
        (axes[0], 'delta_mae_loglife', 'ΔMAE by life bin'),
        (axes[1], 'delta_rmse_loglife', 'ΔRMSE by life bin'),
    ]:
        for (ablation_fam, baseline_fam), g in paired_headfeat_comparison.groupby(['model_family_ablation', 'model_family_baseline']):
            d = g.set_index('life_bin').reindex(life_bin_order).reset_index()
            x = np.arange(len(d), dtype=float)
            ax.plot(x, d[delta_col], marker='o', label=f'{ablation_fam} − {baseline_fam}')
        ax.axhline(0, color='k', lw=1.0)
        ax.set_xticks(np.arange(len(life_bin_order)))
        ax.set_xticklabels(life_bin_order, rotation=20, ha='right')
        ax.set_title(title)
        ax.set_ylabel('decades')
        ax.grid(axis='y', alpha=0.3)
    axes[0].legend(fontsize=7)
    fig.suptitle(f'Engineered-feature ablation by life bin{chr(10)}Negative delta = engineered-feature variant lower error')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'figure_headfeat_vs_baseline_by_life_bin.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURES_DIR / 'figure_headfeat_vs_baseline_by_life_bin.pdf', bbox_inches='tight')
    plt.close(fig)

if paired_headfeat_comparison.empty:
    display(Markdown('**No paired headfeat vs baseline comparison available.**'))
else:
    display(Markdown(f'**Paired headfeat vs baseline comparison** saved to `{RESULTS_DIR}/paired_headfeat_vs_baseline_by_life_bin.csv`.'))
    display(paired_headfeat_comparison[['model_family_ablation', 'model_family_baseline', 'life_bin', 'delta_mae_loglife', 'delta_rmse_loglife', 'note_delta_sign']])


In [ ]:

artifact_list = [
    'summary_table.csv',
    'life_band_metrics.csv',
    'grouped_region_metrics.csv',
    'geometry_summary.csv',
    'paired_summary.csv',
    'feature_inference_validation.csv',
    'paired_headfeat_vs_baseline.csv',
    'paired_headfeat_vs_baseline_by_life_bin.csv',
    'run_metadata.json',
    'figure_main_metrics.pdf/png',
    'figure_life_bands.pdf/png',
    'figure_grouped_regions.pdf/png',
    'figure_headfeat_vs_baseline_delta.pdf/png',
    'figure_headfeat_vs_baseline_by_life_bin.pdf/png',
    'notebook_summary.md',
]
display(pd.DataFrame({'artifact': artifact_list}))


## Final note


In [ ]:

summary_lines = [
    '# Engineered geometric features notebook summary',
    '',
    f'Repository commit: `{COMMIT}`',
    '',
    'Evaluation label: **validation-split evaluation**.',
    '',
    'ArGEnT engineered-feature inference was validated against the training-script `INPUT_COLS` / `QUERY_COLS` metadata.',
    'PointNet headfeat inference now selects engineered features by explicit checkpoint column IDs instead of positional slicing, preserving training-time feature identity and normalization order.',
    'The saved `feature_inference_validation.csv` report records feature counts, names, tensor shape, dtype, finite-value checks, preprocessing artifact references, and sample-alignment status.',
    '',
]
if not dataset_available:
    summary_lines.extend([
        'Dataset limitation: the quantitative zonal HDF5 asset is unavailable in this environment, so the notebook completed with metadata-only validation artifacts and without regenerated numerical figures.',
        '',
    ])
else:
    summary_lines.extend([
        'Dataset available: quantitative feature-aware inference executed on the validation split and the resulting metrics/figures were regenerated from aligned shared sample IDs.',
        '',
    ])
summary_lines.extend([
    'Static audit finding: no evidence was found that ArGEnT itself was dropping engineered inputs; its checkpoint/training metadata requires the engineered columns to enter through the encoder token tensor.',
    'A separate notebook-side fragility was fixed for PointNet headfeat inference so engineered features are now addressed by their true training column IDs and not by positional prefix slicing.',
])
summary_text = chr(10).join(summary_lines)
(RESULTS_DIR / 'notebook_summary.md').write_text(summary_text, encoding='utf-8')
display(Markdown(summary_text))
